# Policy Gradient Methods
## REINFORCE: Direct Policy Optimization for Reinforcement Learning

This notebook provides a self-contained, from-scratch implementation of **Policy Gradient** methods for reinforcement learning. We implement the REINFORCE algorithm with multiple variance-reduction techniques and apply it to both discrete (CartPole) and continuous (Pendulum) control tasks using PyTorch. Every concept is developed from first principles with detailed derivations and visualizations.

**What you'll learn:**
1. Policy gradient theorem and its derivation using the log-derivative trick
2. REINFORCE algorithm: the simplest policy gradient method
3. Variance reduction techniques: baselines and reward-to-go
4. Continuous policy parameterization with Gaussian policies
5. Discrete (CartPole) and continuous (Pendulum) control experiments

**Prerequisites:** Neural networks, probability (expectations, log-derivative trick), PyTorch basics.

**References:**
- Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd Ed., MIT Press, 2018, Chapter 13.
- Williams, R.J., *Simple Statistical Gradient-Following Algorithms for Connectionist Reinforcement Learning*, Machine Learning, 1992.
- Sutton, R.S., McAllester, D., Singh, S., Mansour, Y., *Policy Gradient Methods for Reinforcement Learning with Function Approximation*, NeurIPS, 2000.

---
## 1. Imports and Configuration

In [ ]:
# ============================================================
#  Imports and Configuration
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical, Normal
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ---- reproducibility ----
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- plotting defaults ----
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

# ---- color palette ----
COLORS = {
    'basic': 'steelblue',
    'baseline': 'coral',
    'rtg': 'seagreen',
    'continuous': 'goldenrod',
    'extra': 'mediumpurple',
}

# ---- hyperparameters ----
GAMMA = 0.99           # discount factor
LR_DISCRETE = 1e-3     # learning rate for discrete policy
LR_CONTINUOUS = 3e-4   # learning rate for continuous policy
N_EPISODES = 2000      # training episodes
HIDDEN_DIM = 128       # hidden layer size

# ---- device ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Gymnasium version: {gym.__version__}")

---
## 2. Policy Parameterization

In policy gradient methods, we directly parameterize the policy $\pi_\theta(a|s)$ using a neural network with parameters $\theta$. This is in contrast to value-based methods (Q-learning, SARSA) that derive a policy implicitly from learned value functions.

**Why direct policy parameterization?**
- Can represent **stochastic** policies naturally (essential for partial observability and exploration)
- Works in **continuous** action spaces without discretization
- Policy changes **smoothly** with parameter updates (small $\Delta\theta \Rightarrow$ small change in $\pi_\theta$)
- Can incorporate domain knowledge through policy structure

**Discrete actions (softmax policy):** For environments with $|\mathcal{A}|$ discrete actions:

$$\pi_\theta(a|s) = \frac{\exp(h_\theta(s, a))}{\sum_{a'} \exp(h_\theta(s, a'))}$$

where $h_\theta(s, a)$ are logits from a neural network.

**Continuous actions (Gaussian policy):** For continuous action spaces:

$$\pi_\theta(a|s) = \mathcal{N}(a \mid \mu_\theta(s), \sigma_\theta(s)^2)$$

where $\mu_\theta(s)$ and $\sigma_\theta(s)$ are the mean and standard deviation output by the network.

---
## 3. Policy Gradient Theorem

**Objective:** Maximize the expected cumulative reward under policy $\pi_\theta$:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)] = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \gamma^t r_t\right]$$

**Derivation using the log-derivative trick:**

The gradient of the objective is:

$$\nabla_\theta J(\theta) = \nabla_\theta \int p_\theta(\tau) R(\tau) \, d\tau$$

We cannot easily differentiate through the sampling process, but using the identity $\nabla_\theta p_\theta(\tau) = p_\theta(\tau) \nabla_\theta \log p_\theta(\tau)$:

$$\nabla_\theta J(\theta) = \int p_\theta(\tau) \nabla_\theta \log p_\theta(\tau) \cdot R(\tau) \, d\tau = \mathbb{E}_{\tau \sim \pi_\theta}\left[\nabla_\theta \log p_\theta(\tau) \cdot R(\tau)\right]$$

Since $\log p_\theta(\tau) = \log p(s_0) + \sum_{t=0}^{T} \left[\log \pi_\theta(a_t|s_t) + \log p(s_{t+1}|s_t, a_t)\right]$, and only $\log \pi_\theta$ depends on $\theta$:

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]}$$

where $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$ is the return from time step $t$.

---
## 4. Variance Reduction

The basic policy gradient estimator has **high variance** because it weights log-probabilities by full trajectory returns. Three key techniques reduce this variance:

### 4.1 Baselines

Subtracting a baseline $b(s_t)$ from the return does not change the expectation but can dramatically reduce variance:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot (G_t - b(s_t))\right]$$

**Proof that baseline doesn't bias the gradient:**

$$\mathbb{E}_{\tau}\left[\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot b(s_t)\right] = \mathbb{E}_{s_t}\left[b(s_t) \sum_{a} \pi_\theta(a|s_t) \nabla_\theta \log \pi_\theta(a|s_t)\right] = \mathbb{E}_{s_t}\left[b(s_t) \nabla_\theta \sum_{a} \pi_\theta(a|s_t)\right] = 0$$

The optimal baseline is $b^*(s_t) \approx V^\pi(s_t)$, the state-value function.

### 4.2 Reward-to-Go

Actions at time $t$ can only affect future rewards. Instead of using the full trajectory return $R(\tau)$, we use only the **future return** from step $t$:

$$G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$$

This removes the contribution of past rewards which just add noise.

### 4.3 State-Value Baseline

Learn a value function $V_\phi(s)$ as the baseline. The advantage $A_t = G_t - V_\phi(s_t)$ indicates how much better an action was compared to the average.

---
## 5. REINFORCE Algorithm

**Algorithm: REINFORCE (Monte Carlo Policy Gradient)**

```
Input: differentiable policy π_θ, learning rate α, discount γ
Initialize: policy parameters θ randomly

Repeat for each episode:
    1. Generate trajectory τ = (s₀, a₀, r₀, ..., s_T) following π_θ
    2. For each time step t = 0, 1, ..., T:
       a. Compute return G_t = Σ_{k=t}^{T} γ^{k-t} r_k
       b. Compute policy gradient: ∇_θ log π_θ(a_t|s_t) · G_t
    3. Update: θ ← θ + α Σ_t ∇_θ log π_θ(a_t|s_t) · G_t
```

**With Baseline:**
```
Additionally maintain value function V_φ(s)
    2b. Compute advantage: A_t = G_t - V_φ(s_t)
    2c. Policy gradient uses A_t instead of G_t
    3b. Update baseline: φ ← φ - β ∇_φ (G_t - V_φ(s_t))²
```

---
## 6. Network Architectures

In [ ]:
# ============================================================
#  Discrete Policy Network (Softmax)
# ============================================================

class DiscretePolicy(nn.Module):
    """Policy network for discrete action spaces.
    
    Architecture: state -> FC -> ReLU -> FC -> ReLU -> FC -> softmax
    Outputs a categorical distribution over actions.
    """
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = HIDDEN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        """Returns action probabilities via softmax."""
        logits = self.net(state)
        return F.softmax(logits, dim=-1)
    
    def select_action(self, state: np.ndarray) -> Tuple[int, torch.Tensor]:
        """Sample an action from the policy and return (action, log_prob)."""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs = self.forward(state_t)
        dist = Categorical(probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob


# ============================================================
#  Continuous Policy Network (Gaussian)
# ============================================================

class ContinuousPolicy(nn.Module):
    """Policy network for continuous action spaces.
    
    Architecture: state -> FC -> ReLU -> FC -> ReLU -> (mean, log_std)
    Outputs a Gaussian distribution parameterized by mean and std.
    """
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = HIDDEN_DIM,
                 action_low: float = -2.0, action_high: float = 2.0):
        super().__init__()
        self.action_low = action_low
        self.action_high = action_high
        
        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        # Mean and log_std heads
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state: torch.Tensor) -> Normal:
        """Returns a Normal distribution over actions."""
        features = self.shared(state)
        mean = self.mean_head(features)
        log_std = self.log_std_head(features)
        # Clamp log_std for numerical stability
        log_std = torch.clamp(log_std, min=-20.0, max=2.0)
        std = log_std.exp()
        return Normal(mean, std)
    
    def select_action(self, state: np.ndarray) -> Tuple[np.ndarray, torch.Tensor]:
        """Sample an action and return (clipped_action, log_prob)."""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        dist = self.forward(state_t)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(dim=-1)  # sum over action dims
        # Clip to valid action range
        action_clipped = torch.clamp(action, self.action_low, self.action_high)
        return action_clipped.cpu().detach().numpy().flatten(), log_prob


# ============================================================
#  Value Baseline Network
# ============================================================

class ValueBaseline(nn.Module):
    """State-value function V(s) used as a baseline for variance reduction."""
    
    def __init__(self, state_dim: int, hidden_dim: int = HIDDEN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        """Predict state value V(s)."""
        return self.net(state).squeeze(-1)


print("Network architectures defined.")
# Quick sanity check
_dp = DiscretePolicy(4, 2)
_cp = ContinuousPolicy(3, 1)
_vb = ValueBaseline(4)
print(f"DiscretePolicy params:  {sum(p.numel() for p in _dp.parameters()):,}")
print(f"ContinuousPolicy params: {sum(p.numel() for p in _cp.parameters()):,}")
print(f"ValueBaseline params:   {sum(p.numel() for p in _vb.parameters()):,}")

---
## 7. Utility Functions

In [ ]:
# ============================================================
#  Utility Functions
# ============================================================

def compute_returns(rewards: List[float], gamma: float) -> List[float]:
    """Compute discounted returns G_t = sum_{k=t}^{T} gamma^{k-t} * r_k.
    
    Uses reverse accumulation for O(T) efficiency.
    """
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns


def smooth(data: List[float], window: int = 50) -> np.ndarray:
    """Smooth a 1D signal using a running average."""
    if len(data) < window:
        return np.array(data)
    kernel = np.ones(window) / window
    return np.convolve(data, kernel, mode='valid')


def set_all_seeds(seed: int):
    """Set seeds for reproducibility across all libraries."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Quick verification of compute_returns
test_rewards = [1.0, 1.0, 1.0]
test_returns = compute_returns(test_rewards, gamma=0.99)
expected = [1 + 0.99 + 0.99**2, 1 + 0.99, 1.0]
assert all(abs(a - b) < 1e-10 for a, b in zip(test_returns, expected)), "compute_returns failed"
print(f"compute_returns test: rewards={test_rewards} -> returns={[round(r,4) for r in test_returns]}")
print("Utility functions ready.")

---
## 8. REINFORCE — Basic Implementation

In [ ]:
# ============================================================
#  REINFORCE — Basic (no baseline, full trajectory return)
# ============================================================

def reinforce_basic(env_name: str, n_episodes: int = N_EPISODES,
                    gamma: float = GAMMA, lr: float = LR_DISCRETE,
                    seed: int = SEED) -> Tuple[List[float], DiscretePolicy, List[List[float]]]:
    """Train a discrete policy using basic REINFORCE.
    
    Uses full trajectory return R(tau) to weight all log-probs equally.
    Returns: (episode_rewards, trained_policy, per_episode_grad_norms)
    """
    set_all_seeds(seed)
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    policy = DiscretePolicy(state_dim, action_dim).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    
    episode_rewards = []
    grad_norms_history = []  # track gradient norms for variance analysis
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        log_probs = []
        rewards = []
        done = False
        
        # ---- collect trajectory ----
        while not done:
            action, log_prob = policy.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            rewards.append(reward)
            state = next_state
        
        episode_rewards.append(sum(rewards))
        
        # ---- compute returns (basic: use full trajectory return for all steps) ----
        returns = compute_returns(rewards, gamma)
        returns_t = torch.FloatTensor(returns).to(device)
        # Normalize returns for stability
        if len(returns_t) > 1:
            returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)
        
        # ---- policy gradient update ----
        policy_loss = []
        for lp, G in zip(log_probs, returns_t):
            policy_loss.append(-lp * G)  # negative because we maximize
        
        optimizer.zero_grad()
        loss = torch.stack(policy_loss).sum()
        loss.backward()
        
        # Record gradient norms
        grad_norms = []
        for p in policy.parameters():
            if p.grad is not None:
                grad_norms.append(p.grad.norm().item())
        grad_norms_history.append(grad_norms)
        
        optimizer.step()
        
        if (ep + 1) % 500 == 0:
            avg = np.mean(episode_rewards[-100:])
            print(f"  Episode {ep+1}/{n_episodes} | Avg Reward (last 100): {avg:.1f}")
    
    env.close()
    return episode_rewards, policy, grad_norms_history


print("REINFORCE (basic) defined.")

---
## 9. REINFORCE — With Baseline

In [ ]:
# ============================================================
#  REINFORCE — With Learned Value Baseline
# ============================================================

def reinforce_baseline(env_name: str, n_episodes: int = N_EPISODES,
                       gamma: float = GAMMA, lr: float = LR_DISCRETE,
                       seed: int = SEED) -> Tuple[List[float], DiscretePolicy, List[List[float]]]:
    """Train a discrete policy using REINFORCE with a learned value baseline.
    
    The advantage A_t = G_t - V(s_t) is used instead of raw returns.
    Returns: (episode_rewards, trained_policy, per_episode_grad_norms)
    """
    set_all_seeds(seed)
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    policy = DiscretePolicy(state_dim, action_dim).to(device)
    baseline = ValueBaseline(state_dim).to(device)
    policy_optimizer = optim.Adam(policy.parameters(), lr=lr)
    baseline_optimizer = optim.Adam(baseline.parameters(), lr=lr)
    
    episode_rewards = []
    grad_norms_history = []
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        log_probs = []
        rewards = []
        states = []
        done = False
        
        # ---- collect trajectory ----
        while not done:
            states.append(state)
            action, log_prob = policy.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            rewards.append(reward)
            state = next_state
        
        episode_rewards.append(sum(rewards))
        
        # ---- compute returns ----
        returns = compute_returns(rewards, gamma)
        returns_t = torch.FloatTensor(returns).to(device)
        states_t = torch.FloatTensor(np.array(states)).to(device)
        
        # ---- compute advantages: A_t = G_t - V(s_t) ----
        values = baseline(states_t).detach()
        advantages = returns_t - values
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # ---- policy gradient update ----
        policy_loss = []
        for lp, adv in zip(log_probs, advantages):
            policy_loss.append(-lp * adv)
        
        policy_optimizer.zero_grad()
        p_loss = torch.stack(policy_loss).sum()
        p_loss.backward()
        
        # Record gradient norms
        grad_norms = []
        for p in policy.parameters():
            if p.grad is not None:
                grad_norms.append(p.grad.norm().item())
        grad_norms_history.append(grad_norms)
        
        policy_optimizer.step()
        
        # ---- baseline (value function) update ----
        baseline_optimizer.zero_grad()
        values_pred = baseline(states_t)
        baseline_loss = F.mse_loss(values_pred, returns_t)
        baseline_loss.backward()
        baseline_optimizer.step()
        
        if (ep + 1) % 500 == 0:
            avg = np.mean(episode_rewards[-100:])
            print(f"  Episode {ep+1}/{n_episodes} | Avg Reward (last 100): {avg:.1f}")
    
    env.close()
    return episode_rewards, policy, grad_norms_history


print("REINFORCE (with baseline) defined.")

---
## 10. REINFORCE — Reward-to-Go

In [ ]:
# ============================================================
#  REINFORCE — Reward-to-Go (causality-aware)
# ============================================================

def reinforce_reward_to_go(env_name: str, n_episodes: int = N_EPISODES,
                           gamma: float = GAMMA, lr: float = LR_DISCRETE,
                           seed: int = SEED) -> Tuple[List[float], DiscretePolicy, List[List[float]]]:
    """Train a discrete policy using REINFORCE with reward-to-go.
    
    Each log_prob is weighted only by future returns G_t, not the full R(tau).
    This exploits causality: actions at time t cannot affect rewards before t.
    Returns: (episode_rewards, trained_policy, per_episode_grad_norms)
    """
    set_all_seeds(seed)
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    policy = DiscretePolicy(state_dim, action_dim).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    
    episode_rewards = []
    grad_norms_history = []
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        log_probs = []
        rewards = []
        done = False
        
        # ---- collect trajectory ----
        while not done:
            action, log_prob = policy.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            rewards.append(reward)
            state = next_state
        
        episode_rewards.append(sum(rewards))
        
        # ---- compute reward-to-go returns ----
        returns = compute_returns(rewards, gamma)
        returns_t = torch.FloatTensor(returns).to(device)
        if len(returns_t) > 1:
            returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)
        
        # ---- policy gradient update ----
        policy_loss = []
        for lp, G in zip(log_probs, returns_t):
            policy_loss.append(-lp * G)
        
        optimizer.zero_grad()
        loss = torch.stack(policy_loss).sum()
        loss.backward()
        
        # Record gradient norms
        grad_norms = []
        for p in policy.parameters():
            if p.grad is not None:
                grad_norms.append(p.grad.norm().item())
        grad_norms_history.append(grad_norms)
        
        optimizer.step()
        
        if (ep + 1) % 500 == 0:
            avg = np.mean(episode_rewards[-100:])
            print(f"  Episode {ep+1}/{n_episodes} | Avg Reward (last 100): {avg:.1f}")
    
    env.close()
    return episode_rewards, policy, grad_norms_history


print("REINFORCE (reward-to-go) defined.")

---
## 11. REINFORCE — Continuous Control (Gaussian Policy)

In [ ]:
# ============================================================
#  REINFORCE — Continuous Gaussian Policy with Baseline
# ============================================================

def reinforce_continuous(env_name: str, n_episodes: int = N_EPISODES,
                         gamma: float = GAMMA, lr: float = LR_CONTINUOUS,
                         seed: int = SEED) -> Tuple[List[float], ContinuousPolicy, List[float]]:
    """Train a continuous Gaussian policy using REINFORCE with baseline.
    
    Applied to environments with continuous action spaces (e.g., Pendulum-v1).
    Returns: (episode_rewards, trained_policy, policy_losses)
    """
    set_all_seeds(seed)
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    action_low = float(env.action_space.low[0])
    action_high = float(env.action_space.high[0])
    
    policy = ContinuousPolicy(state_dim, action_dim,
                              action_low=action_low, action_high=action_high).to(device)
    baseline = ValueBaseline(state_dim).to(device)
    policy_optimizer = optim.Adam(policy.parameters(), lr=lr)
    baseline_optimizer = optim.Adam(baseline.parameters(), lr=lr)
    
    episode_rewards = []
    policy_losses = []
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        log_probs = []
        rewards = []
        states = []
        done = False
        
        # ---- collect trajectory ----
        while not done:
            states.append(state)
            action, log_prob = policy.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            rewards.append(reward)
            state = next_state
        
        episode_rewards.append(sum(rewards))
        
        # ---- compute returns and advantages ----
        returns = compute_returns(rewards, gamma)
        returns_t = torch.FloatTensor(returns).to(device)
        states_t = torch.FloatTensor(np.array(states)).to(device)
        
        values = baseline(states_t).detach()
        advantages = returns_t - values
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # ---- policy update ----
        policy_loss = []
        for lp, adv in zip(log_probs, advantages):
            policy_loss.append(-lp * adv)
        
        policy_optimizer.zero_grad()
        p_loss = torch.stack(policy_loss).sum()
        p_loss.backward()
        # Gradient clipping for stability in continuous control
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        policy_optimizer.step()
        policy_losses.append(p_loss.item())
        
        # ---- baseline update ----
        baseline_optimizer.zero_grad()
        values_pred = baseline(states_t)
        baseline_loss = F.mse_loss(values_pred, returns_t)
        baseline_loss.backward()
        baseline_optimizer.step()
        
        if (ep + 1) % 500 == 0:
            avg = np.mean(episode_rewards[-100:])
            print(f"  Episode {ep+1}/{n_episodes} | Avg Reward (last 100): {avg:.1f}")
    
    env.close()
    return episode_rewards, policy, policy_losses


print("REINFORCE (continuous) defined.")

---
## 12. Experiment 1 — CartPole with Basic REINFORCE

In [ ]:
# ============================================================
#  Experiment 1: CartPole-v1 with Basic REINFORCE
# ============================================================

print("Training basic REINFORCE on CartPole-v1...")
rewards_basic, policy_basic, grads_basic = reinforce_basic('CartPole-v1')
print(f"Final avg reward (last 100): {np.mean(rewards_basic[-100:]):.1f}")

---
## 13. Experiment 2 — CartPole with Baseline

In [ ]:
# ============================================================
#  Experiment 2: CartPole-v1 with REINFORCE + Baseline
# ============================================================

print("Training REINFORCE with baseline on CartPole-v1...")
rewards_baseline, policy_baseline, grads_baseline = reinforce_baseline('CartPole-v1')
print(f"Final avg reward (last 100): {np.mean(rewards_baseline[-100:]):.1f}")

---
## 14. Experiment 3 — CartPole with Reward-to-Go

In [ ]:
# ============================================================
#  Experiment 3: CartPole-v1 with REINFORCE + Reward-to-Go
# ============================================================

print("Training REINFORCE with reward-to-go on CartPole-v1...")
rewards_rtg, policy_rtg, grads_rtg = reinforce_reward_to_go('CartPole-v1')
print(f"Final avg reward (last 100): {np.mean(rewards_rtg[-100:]):.1f}")

---
## 15. Visualization 1 — Training Reward Curves

In [ ]:
# ============================================================
#  Visualization 1: Training Reward Curves (all 3 variants)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ---- Raw rewards ----
ax = axes[0]
alpha_raw = 0.15
ax.plot(rewards_basic, alpha=alpha_raw, color=COLORS['basic'])
ax.plot(rewards_baseline, alpha=alpha_raw, color=COLORS['baseline'])
ax.plot(rewards_rtg, alpha=alpha_raw, color=COLORS['rtg'])
ax.set_title('Raw Episode Rewards')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend(['Basic', 'Baseline', 'Reward-to-Go'], loc='lower right')

# ---- Smoothed rewards ----
ax = axes[1]
window = 50
s_basic = smooth(rewards_basic, window)
s_baseline = smooth(rewards_baseline, window)
s_rtg = smooth(rewards_rtg, window)

ax.plot(s_basic, color=COLORS['basic'], label='Basic REINFORCE')
ax.plot(s_baseline, color=COLORS['baseline'], label='With Baseline')
ax.plot(s_rtg, color=COLORS['rtg'], label='Reward-to-Go')
ax.axhline(y=195, color='gray', linestyle='--', alpha=0.7, label='Solved threshold (195)')
ax.set_title(f'Smoothed Rewards (window={window})')
ax.set_xlabel('Episode')
ax.set_ylabel('Avg Reward')
ax.legend(loc='lower right')

plt.suptitle('CartPole-v1: REINFORCE Variants Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 16. Visualization 2 — Gradient Variance Comparison

In [ ]:
# ============================================================
#  Visualization 2: Gradient Variance Comparison
# ============================================================

def compute_grad_variance(grad_norms_history: List[List[float]], window: int = 100) -> np.ndarray:
    """Compute rolling variance of gradient norms (averaged over parameters)."""
    # Average gradient norm across parameters for each episode
    avg_norms = [np.mean(gn) if len(gn) > 0 else 0.0 for gn in grad_norms_history]
    # Compute rolling variance
    variances = []
    for i in range(len(avg_norms)):
        start = max(0, i - window + 1)
        variances.append(np.var(avg_norms[start:i+1]))
    return np.array(variances)


var_basic = compute_grad_variance(grads_basic)
var_baseline = compute_grad_variance(grads_baseline)
var_rtg = compute_grad_variance(grads_rtg)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ---- Gradient norm over episodes ----
ax = axes[0]
avg_basic = smooth([np.mean(gn) for gn in grads_basic], 50)
avg_baseline_g = smooth([np.mean(gn) for gn in grads_baseline], 50)
avg_rtg_g = smooth([np.mean(gn) for gn in grads_rtg], 50)
ax.plot(avg_basic, color=COLORS['basic'], label='Basic')
ax.plot(avg_baseline_g, color=COLORS['baseline'], label='Baseline')
ax.plot(avg_rtg_g, color=COLORS['rtg'], label='Reward-to-Go')
ax.set_title('Smoothed Gradient Norm')
ax.set_xlabel('Episode')
ax.set_ylabel('Avg Gradient Norm')
ax.legend()

# ---- Gradient variance over episodes ----
ax = axes[1]
ax.plot(smooth(var_basic.tolist(), 50), color=COLORS['basic'], label='Basic')
ax.plot(smooth(var_baseline.tolist(), 50), color=COLORS['baseline'], label='Baseline')
ax.plot(smooth(var_rtg.tolist(), 50), color=COLORS['rtg'], label='Reward-to-Go')
ax.set_title('Rolling Gradient Variance (window=100)')
ax.set_xlabel('Episode')
ax.set_ylabel('Variance of Gradient Norms')
ax.legend()

plt.suptitle('Gradient Analysis: Effect of Variance Reduction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 17. Visualization 3 — Policy Distribution Evolution

In [ ]:
# ============================================================
#  Visualization 3: Policy Distribution Evolution
# ============================================================
#
# We retrain a short run and snapshot the softmax probabilities at
# selected episodes to see how the policy evolves.
# ============================================================

def train_with_snapshots(env_name: str, snapshot_episodes: List[int],
                         n_episodes: int = 1000, gamma: float = GAMMA,
                         lr: float = LR_DISCRETE, seed: int = SEED
                         ) -> Tuple[List[float], dict]:
    """Train and snapshot policy distributions at specified episodes."""
    set_all_seeds(seed)
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    policy = DiscretePolicy(state_dim, action_dim).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    
    # Fixed test states for consistent comparison
    test_states = [
        np.array([0.0, 0.0, 0.0, 0.0]),       # centered, balanced
        np.array([0.5, 0.5, 0.1, 0.1]),        # slightly right, tilting right
        np.array([-0.5, -0.5, -0.1, -0.1]),    # slightly left, tilting left
        np.array([0.0, 0.0, 0.15, 1.0]),       # strong rightward tilt
    ]
    state_labels = ['Centered', 'Right-leaning', 'Left-leaning', 'Falling right']
    
    snapshots = {ep: {} for ep in snapshot_episodes}
    episode_rewards = []
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        log_probs = []
        rewards = []
        done = False
        
        while not done:
            action, log_prob = policy.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            rewards.append(reward)
            state = next_state
        
        episode_rewards.append(sum(rewards))
        
        returns = compute_returns(rewards, gamma)
        returns_t = torch.FloatTensor(returns).to(device)
        if len(returns_t) > 1:
            returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)
        
        optimizer.zero_grad()
        loss = sum(-lp * G for lp, G in zip(log_probs, returns_t))
        loss.backward()
        optimizer.step()
        
        # Snapshot policy
        if ep in snapshot_episodes:
            policy.eval()
            with torch.no_grad():
                for i, ts in enumerate(test_states):
                    ts_t = torch.FloatTensor(ts).unsqueeze(0).to(device)
                    probs = policy(ts_t).cpu().numpy().flatten()
                    snapshots[ep][state_labels[i]] = probs.copy()
            policy.train()
    
    env.close()
    return episode_rewards, snapshots


snapshot_eps = [0, 100, 300, 500, 800, 999]
print("Training with snapshots...")
snap_rewards, snapshots = train_with_snapshots('CartPole-v1', snapshot_eps)

# ---- Plot policy evolution ----
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
state_labels = ['Centered', 'Right-leaning', 'Left-leaning', 'Falling right']
bar_colors = [COLORS['basic'], COLORS['baseline']]
action_names = ['Left', 'Right']

for idx, ep in enumerate(snapshot_eps):
    ax = axes[idx // 3, idx % 3]
    x = np.arange(len(state_labels))
    width = 0.35
    
    left_probs = [snapshots[ep][sl][0] for sl in state_labels]
    right_probs = [snapshots[ep][sl][1] for sl in state_labels]
    
    ax.bar(x - width/2, left_probs, width, label='P(Left)', color=bar_colors[0], alpha=0.8)
    ax.bar(x + width/2, right_probs, width, label='P(Right)', color=bar_colors[1], alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(state_labels, rotation=30, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title(f'Episode {ep}', fontsize=11)
    ax.set_ylabel('Probability')
    if idx == 0:
        ax.legend(fontsize=9)

plt.suptitle('Policy Distribution Evolution Over Training (CartPole)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 18. Experiment 4 — Pendulum with Continuous Gaussian Policy

In [ ]:
# ============================================================
#  Experiment 4: Pendulum-v1 with Continuous Policy
# ============================================================

print("Training continuous REINFORCE on Pendulum-v1...")
rewards_pendulum, policy_pendulum, losses_pendulum = reinforce_continuous('Pendulum-v1')
print(f"Final avg reward (last 100): {np.mean(rewards_pendulum[-100:]):.1f}")

---
## 19. Visualization 4 — Continuous Policy: Mean Action and Std Bands

In [ ]:
# ============================================================
#  Visualization 4: Continuous Policy — Mean ± Std over State
# ============================================================

# Evaluate the trained continuous policy over a grid of states.
# Pendulum state: [cos(theta), sin(theta), theta_dot]
# We sweep theta from -pi to pi, keeping theta_dot = 0.

thetas = np.linspace(-np.pi, np.pi, 200)
means = []
stds = []

policy_pendulum.eval()
with torch.no_grad():
    for theta in thetas:
        state = np.array([np.cos(theta), np.sin(theta), 0.0])
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        dist = policy_pendulum(state_t)
        means.append(dist.mean.cpu().numpy().flatten()[0])
        stds.append(dist.stddev.cpu().numpy().flatten()[0])

means = np.array(means)
stds = np.array(stds)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ---- Mean action with std bands ----
ax = axes[0]
ax.plot(np.degrees(thetas), means, color=COLORS['continuous'], label='Mean action $\\mu(s)$')
ax.fill_between(np.degrees(thetas), means - stds, means + stds,
                alpha=0.25, color=COLORS['continuous'], label='$\\pm 1\\sigma$')
ax.fill_between(np.degrees(thetas), means - 2*stds, means + 2*stds,
                alpha=0.1, color=COLORS['continuous'], label='$\\pm 2\\sigma$')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Angle $\\theta$ (degrees)')
ax.set_ylabel('Torque')
ax.set_title('Learned Gaussian Policy: Mean Action vs Angle')
ax.legend(loc='upper right')

# ---- Std as function of state ----
ax = axes[1]
ax.plot(np.degrees(thetas), stds, color=COLORS['extra'], linewidth=2)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Angle $\\theta$ (degrees)')
ax.set_ylabel('Standard Deviation $\\sigma(s)$')
ax.set_title('Policy Uncertainty vs Angle')

plt.suptitle('Pendulum: Learned Continuous Policy Landscape', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 20. Visualization 5 — Loss Curves

In [ ]:
# ============================================================
#  Visualization 5: Loss Curves
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(12, 5))

smoothed_loss = smooth(losses_pendulum, window=50)
ax.plot(losses_pendulum, alpha=0.15, color=COLORS['continuous'])
ax.plot(np.arange(len(smoothed_loss)) + 25, smoothed_loss,
        color=COLORS['continuous'], label='Smoothed Policy Loss')
ax.set_xlabel('Episode')
ax.set_ylabel('Policy Loss')
ax.set_title('Pendulum: Policy Gradient Loss Over Training', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 21. Visualization 6 — Pendulum Reward Curve

In [ ]:
# ============================================================
#  Visualization 6: Pendulum Reward Curve
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(12, 5))

ax.plot(rewards_pendulum, alpha=0.15, color=COLORS['continuous'])
smoothed_pend = smooth(rewards_pendulum, window=50)
ax.plot(np.arange(len(smoothed_pend)) + 25, smoothed_pend,
        color=COLORS['continuous'], linewidth=2.5, label='Smoothed (window=50)')
ax.axhline(y=-500, color='gray', linestyle='--', alpha=0.7, label='Target threshold (-500)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Pendulum-v1: Continuous REINFORCE Training Curve', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 22. Verification

In [ ]:
# ============================================================
#  Verification: Automated Checks
# ============================================================

print("=" * 60)
print("  VERIFICATION RESULTS")
print("=" * 60)

# ---- Check 1: REINFORCE solves CartPole (avg >= 195) ----
avg_basic_final = np.mean(rewards_basic[-100:])
avg_baseline_final = np.mean(rewards_baseline[-100:])
avg_rtg_final = np.mean(rewards_rtg[-100:])
best_cartpole = max(avg_basic_final, avg_baseline_final, avg_rtg_final)
cartpole_solved = best_cartpole >= 195.0
tag1 = "[PASS]" if cartpole_solved else "[FAIL]"
print(f"\n{tag1} REINFORCE solves CartPole-v1 (best avg >= 195)")
print(f"       Basic: {avg_basic_final:.1f} | Baseline: {avg_baseline_final:.1f} | RTG: {avg_rtg_final:.1f}")

# ---- Check 2: Baseline reduces gradient variance ----
# Compare average gradient variance over last 500 episodes
var_basic_late = np.mean(compute_grad_variance(grads_basic)[-500:])
var_baseline_late = np.mean(compute_grad_variance(grads_baseline)[-500:])
baseline_reduces_var = var_baseline_late < var_basic_late
tag2 = "[PASS]" if baseline_reduces_var else "[FAIL]"
print(f"\n{tag2} Baseline reduces gradient variance")
print(f"       Basic variance: {var_basic_late:.4f} | Baseline variance: {var_baseline_late:.4f}")

# ---- Check 3: Reward-to-go converges faster ----
# Compare episode at which each method first reaches avg reward >= 150
def first_above(rewards, threshold=150, window=50):
    """Find the first episode where smoothed reward exceeds threshold."""
    sm = smooth(rewards, window)
    above = np.where(sm >= threshold)[0]
    return above[0] + window if len(above) > 0 else len(rewards)

ep_basic = first_above(rewards_basic)
ep_rtg = first_above(rewards_rtg)
rtg_faster = ep_rtg <= ep_basic
tag3 = "[PASS]" if rtg_faster else "[FAIL]"
print(f"\n{tag3} Reward-to-go converges faster than basic REINFORCE")
print(f"       Basic first reaches 150 at ep {ep_basic} | RTG at ep {ep_rtg}")

# ---- Check 4: Continuous policy learns pendulum (avg > -500) ----
avg_pendulum_final = np.mean(rewards_pendulum[-100:])
pendulum_learned = avg_pendulum_final > -500
tag4 = "[PASS]" if pendulum_learned else "[FAIL]"
print(f"\n{tag4} Continuous policy learns Pendulum (avg reward > -500)")
print(f"       Final avg reward (last 100): {avg_pendulum_final:.1f}")

print("\n" + "=" * 60)
n_pass = sum([cartpole_solved, baseline_reduces_var, rtg_faster, pendulum_learned])
print(f"  {n_pass}/4 checks passed")
print("=" * 60)

---
## 23. Summary and Key Takeaways

### What We Covered

1. **Policy Gradient Theorem** — We derived the fundamental result that allows us to compute gradients of expected returns without differentiating through the environment dynamics. The key identity $\nabla_\theta \mathbb{E}[f(x)] = \mathbb{E}[f(x) \nabla_\theta \log p_\theta(x)]$ (log-derivative trick / REINFORCE trick) makes this possible.

2. **REINFORCE Algorithm** — The simplest policy gradient method. It collects full episodes, computes Monte Carlo returns, and updates the policy by ascending the gradient $\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t$.

3. **Variance Reduction** — We implemented three key techniques:
   - **Reward-to-go**: Only uses future returns (exploits causality)
   - **Learned baseline**: Subtracts $V(s_t)$ to compute advantages
   - **Return normalization**: Standardizes returns per episode

4. **Discrete vs Continuous Policies** — Softmax parameterization for CartPole (categorical distribution), Gaussian parameterization for Pendulum (normal distribution with learned mean and variance).

### Key Observations

- **High variance** is the central challenge of policy gradient methods. Basic REINFORCE has notoriously noisy gradients.
- **Baselines** are critical: they don't bias the gradient but dramatically reduce variance.
- **Reward-to-go** is a simple but effective improvement — it removes irrelevant past-reward noise.
- **Continuous policies** are harder to train and benefit significantly from gradient clipping and careful learning rate tuning.

### Looking Forward

Policy gradients are the foundation for more advanced algorithms:
- **Actor-Critic**: Replace Monte Carlo returns with TD estimates (lower variance, some bias)
- **PPO/TRPO**: Constrain policy updates for stability (trust region methods)
- **SAC**: Add entropy regularization for exploration in continuous domains
- **A3C/A2C**: Parallel actor-critic for efficiency